# Feature Engineering — Retail Demand Forecasting

Goal:
- Convert sales data into time-series format
- Join calendar and price information
- Create lag, rolling, and calendar-based features

Modeling and evaluation are intentionally excluded.

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

In [2]:
DATA_PATH = "../data/raw/"

sales = pd.read_csv(DATA_PATH + "sales_train_validation.csv")
calendar = pd.read_csv(DATA_PATH + "calendar.csv")
prices = pd.read_csv(DATA_PATH + "sell_prices.csv")

In [3]:
STORE_ID = "CA_1"
sales = sales[sales["store_id"] == STORE_ID]

In [4]:
sales.shape

(3049, 1919)

In [17]:
sales['store_id'].value_counts()

store_id
CA_1    3049
Name: count, dtype: int64

In [15]:
id_cols = [
    "id", "item_id", "dept_id", "cat_id",
    "store_id", "state_id"
]

d_cols = [c for c in sales.columns if c.startswith("d_")]

sales_long = sales.melt(
    id_vars=id_cols,
    value_vars=d_cols,
    var_name="d",
    value_name="sales"
)

In [16]:
sales_long.shape

(5832737, 8)

In [18]:
sales_long.head()   

,id,item_id,dept_id,cat_id,store_id,state_id,d,sales
0,HOBBIES_1_001_CA_1_validation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0
1,HOBBIES_1_002_CA_1_validation,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0
2,HOBBIES_1_003_CA_1_validation,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0
3,HOBBIES_1_004_CA_1_validation,HOBBIES_1_004,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0
4,HOBBIES_1_005_CA_1_validation,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0


In [19]:
calendar_cols = [
    "d", "date", "wm_yr_wk",
    "wday", "month", "year",
    "event_name_1", "event_type_1",
    "event_name_2", "event_type_2",
    "snap_CA"
]

sales_long = sales_long.merge(
    calendar[calendar_cols],
    on="d",
    how="left"
)

In [20]:
sales_long.head()

,id,item_id,dept_id,cat_id,store_id,state_id,d,sales,date,wm_yr_wk,wday,month,year,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA
0,HOBBIES_1_001_CA_1_validation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0,2011-01-29,11101,1,1,2011,NaN,NaN,NaN,NaN,0
1,HOBBIES_1_002_CA_1_validation,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0,2011-01-29,11101,1,1,2011,NaN,NaN,NaN,NaN,0
2,HOBBIES_1_003_CA_1_validation,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0,2011-01-29,11101,1,1,2011,NaN,NaN,NaN,NaN,0
3,HOBBIES_1_004_CA_1_validation,HOBBIES_1_004,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0,2011-01-29,11101,1,1,2011,NaN,NaN,NaN,NaN,0
4,HOBBIES_1_005_CA_1_validation,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0,2011-01-29,11101,1,1,2011,NaN,NaN,NaN,NaN,0


In [21]:
sales_long.isnull().sum().head()

id          0
item_id     0
dept_id     0
cat_id      0
store_id    0
dtype: int64

In [ ]:
sales_long["date"] = pd.to_datetime(sales_long["date"])

# SORT


sales_long = sales_long.sort_values(
    ["item_id", "date"]
).reset_index(drop=True)

In [ ]:
# LAG FEATURES


for lag in [1, 7, 14]:
    sales_long[f"lag_{lag}"] = (
        sales_long
        .groupby("item_id")["sales"]
        .shift(lag)
    )

In [ ]:
# ROLLING MEAN FEATURES


for window in [7, 14]:
    sales_long[f"rolling_mean_{window}"] = (
        sales_long
        .groupby("item_id")["sales"]
        .shift(1)
        .rolling(window)
        .mean()
    )


In [25]:
sales_long["is_weekend"] = sales_long["wday"].isin([1, 7]).astype(int)

In [26]:
sales_long.head()

,id,item_id,dept_id,cat_id,store_id,state_id,d,sales,date,wm_yr_wk,wday,month,year,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,lag_1,lag_7,lag_14,rolling_mean_7,rolling_mean_14,is_weekend
0,FOODS_1_001_CA_1_validation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_1,3,2011-01-29,11101,1,1,2011,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,1
1,FOODS_1_001_CA_1_validation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_2,0,2011-01-30,11101,2,1,2011,NaN,NaN,NaN,NaN,0,3.0,NaN,NaN,NaN,NaN,0
2,FOODS_1_001_CA_1_validation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_3,0,2011-01-31,11101,3,1,2011,NaN,NaN,NaN,NaN,0,0.0,NaN,NaN,NaN,NaN,0
3,FOODS_1_001_CA_1_validation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_4,1,2011-02-01,11101,4,2,2011,NaN,NaN,NaN,NaN,1,0.0,NaN,NaN,NaN,NaN,0
4,FOODS_1_001_CA_1_validation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_5,4,2011-02-02,11101,5,2,2011,NaN,NaN,NaN,NaN,1,1.0,NaN,NaN,NaN,NaN,0


In [27]:
# sales_long.shape
sales_long[["sales", "lag_1", "lag_7", "rolling_mean_7"]].head(10)

,sales,lag_1,lag_7,rolling_mean_7
0,3,NaN,NaN,NaN
1,0,3.0,NaN,NaN
2,0,0.0,NaN,NaN
3,1,0.0,NaN,NaN
4,4,1.0,NaN,NaN
5,2,4.0,NaN,NaN
6,0,2.0,NaN,NaN
7,2,0.0,3.0,1.428571
8,0,2.0,0.0,1.285714
9,0,0.0,0.0,1.285714


In [28]:
sales_long.shape

(5832737, 24)

In [29]:
FEATURE_PATH = "../data/processed/"
sales_long.to_csv(FEATURE_PATH + "features_ca1.csv", index=False)

Notebook 02 completed.
Features created: lag, rolling statistics, and calendar-derived features.
No modeling or splitting performed here by design.